In [1]:
from google.colab import files
from pathlib import Path
import zipfile
import os

print("=== TẢI LÊN VÀ GIẢI NÉN FILE TV1 ===")
uploaded = files.upload()

zip_path = next((Path(name) for name in uploaded if name.lower().endswith(".zip")), None)
if not zip_path:
    zip_path = next(Path('.').glob("*.zip"), None)

if zip_path:
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall("/content/tv1")
    print(f"[+] Giải nén thành công: {zip_path}")
else:
    raise FileNotFoundError("Không tìm thấy file ZIP!")

# Định vị thư mục TV1_DATA_Stroke chính xác
root = Path("/content/tv1/TV1_DATA_Stroke")
if not (root / "src").exists():
    sub_roots = list(Path("/content/tv1").glob("**/TV1_DATA_Stroke"))
    if sub_roots:
        root = sub_roots[0]

print(f"[+] Thư mục dự án sẵn sàng tại: {root}")

=== TẢI LÊN VÀ GIẢI NÉN FILE TV1 ===


Saving TV1_DATA_Hoan_Thanh.zip to TV1_DATA_Hoan_Thanh.zip
[+] Giải nén thành công: TV1_DATA_Hoan_Thanh.zip
[+] Thư mục dự án sẵn sàng tại: /content/tv1/TV1_DATA_Stroke


In [3]:
from pathlib import Path
import sys
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_validate

# Khai báo đường dẫn tuyệt đối chuẩn xác
root = Path("/content/tv1/TV1_DATA_Stroke")
if not (root / "src").exists():
    sub_roots = list(Path("/content/tv1").glob("**/TV1_DATA_Stroke"))
    if sub_roots:
        root = sub_roots[0]

# Ép buộc Python nhận diện thư mục gốc tuyệt đối
sys.path.insert(0, str(root.resolve()))
os.chdir(str(root.resolve()))

# Import an toàn từ kho module của Thành viên 1
from src.model_data import load_partition, make_model_pipeline
from src.data_pipeline import save_csv

print(f"=== BẮT ĐẦU HUẤN LUYỆN 7 MÔ HÌNH (TV2) ===")
def run_tv2_complete_pipeline(core=5, cohort="common"):
    print(f"[*] Đang tải dữ liệu huấn luyện cho CORE {core} (Nhánh: {cohort})...")

    train = load_partition(core=core, cohort=cohort, root=root)
    print(f"[*] Số lượng mẫu train: {len(train.y):,} | Tỷ lệ nhãn đột quỵ: {train.y.mean():.2%}")

    models = {
        "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=1000, random_state=42),
        "SVM": SVC(probability=True, random_state=42),
        "GaussianNaiveBayes": GaussianNB()
    }

    scoring = {
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    }

    cv_results_summary = []
    trained_pipelines = {}

    print("\n[*] Đang chạy 5-fold Cross-Validation chống rò rỉ dữ liệu (Data Leakage)...")
    for name, estimator in models.items():
        scale_age = False if name == "GaussianNaiveBayes" else True
        pipe = make_model_pipeline(estimator, core=core, scale_age=scale_age)

        cv_res = cross_validate(
            pipe, train.X, train.y, cv=train.cv(),
            scoring=scoring, n_jobs=-1, return_train_score=False
        )

        row = {"Model": name}
        for metric in scoring.keys():
            scores = cv_res[f"test_{metric}"]
            row[f"{metric}_mean"] = float(scores.mean())
            row[f"{metric}_std"] = float(scores.std(ddof=1))

        cv_results_summary.append(row)
        print(f"   > {name:20s} | ROC-AUC: {row['roc_auc_mean']:.4f} (±{row['roc_auc_std']:.4f}) | F1: {row['f1_mean']:.4f}")

        param_grids = {
            "LogisticRegression": {"model__C": [0.1, 1.0, 10.0]},
            "RandomForest": {"model__n_estimators": [50, 100], "model__max_depth": [5, 10]},
            "XGBoost": {"model__max_depth": [3, 5], "model__learning_rate": [0.05, 0.1]},
            "DecisionTree": {"model__max_depth": [3, 5, 10]},
            "MLP": {"model__hidden_layer_sizes": [(50,), (100,)], "model__alpha": [0.0001]},
            "SVM": {"model__C": [0.1, 1.0], "model__kernel": ["rbf"]},
            "GaussianNaiveBayes": {}
        }

        grid = GridSearchCV(pipe, param_grid=param_grids[name], cv=train.cv(), scoring="roc_auc", n_jobs=-1)
        grid.fit(train.X, train.y)
        trained_pipelines[name] = grid.best_estimator_

    df_results = pd.DataFrame(cv_results_summary).sort_values(by="roc_auc_mean", ascending=False)

    out_dir = root / "reports" / "tv2_results"
    out_dir.mkdir(parents=True, exist_ok=True)
    results_path = out_dir / f"cv_results_core{core}_{cohort}.csv"
    save_csv(df_results, results_path)
    print(f"\n[+] Đã lưu bảng tổng hợp kết quả CV tại: {results_path}")

    models_dir = root / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    for name, best_pipe in trained_pipelines.items():
        model_file = models_dir / f"{name}_core{core}_{cohort}.joblib"
        joblib.dump(best_pipe, model_file)
    print(f"[+] Đã lưu toàn bộ trọng số mô hình tuning tại thư mục: {models_dir}")

    return df_results, trained_pipelines

# Thực thi chương trình
df_summary, models_dict = run_tv2_complete_pipeline(core=5, cohort="common")

print("\n=== KẾT QUẢ TỔNG HỢP 7 MÔ HÌNH (SẮP XẾP THEO ROC-AUC) ===")
display(df_summary[["Model", "roc_auc_mean", "roc_auc_std", "average_precision_mean", "f1_mean", "accuracy_mean"]])

=== BẮT ĐẦU HUẤN LUYỆN 7 MÔ HÌNH (TV2) ===
[*] Đang tải dữ liệu huấn luyện cho CORE 5 (Nhánh: common)...
[*] Số lượng mẫu train: 2,420 | Tỷ lệ nhãn đột quỵ: 4.96%

[*] Đang chạy 5-fold Cross-Validation chống rò rỉ dữ liệu (Data Leakage)...
   > LogisticRegression   | ROC-AUC: 0.7968 (±0.0234) | F1: 0.0000
   > RandomForest         | ROC-AUC: 0.6503 (±0.0217) | F1: 0.1051
   > XGBoost              | ROC-AUC: 0.7365 (±0.0256) | F1: 0.1010
   > DecisionTree         | ROC-AUC: 0.5688 (±0.0382) | F1: 0.0847
   > MLP                  | ROC-AUC: 0.7816 (±0.0130) | F1: 0.0000
   > SVM                  | ROC-AUC: 0.5996 (±0.0454) | F1: 0.0000
   > GaussianNaiveBayes   | ROC-AUC: 0.7742 (±0.0303) | F1: 0.1388

[+] Đã lưu bảng tổng hợp kết quả CV tại: /content/tv1/TV1_DATA_Stroke/reports/tv2_results/cv_results_core5_common.csv
[+] Đã lưu toàn bộ trọng số mô hình tuning tại thư mục: /content/tv1/TV1_DATA_Stroke/models

=== KẾT QUẢ TỔNG HỢP 7 MÔ HÌNH (SẮP XẾP THEO ROC-AUC) ===


,Model,roc_auc_mean,roc_auc_std,average_precision_mean,f1_mean,accuracy_mean
0,LogisticRegression,0.796833,0.023414,0.159828,0.000000,0.950426
4,MLP,0.781583,0.013035,0.152305,0.000000,0.950426
6,GaussianNaiveBayes,0.774229,0.030273,0.144872,0.138766,0.405349
2,XGBoost,0.736461,0.025617,0.143693,0.101045,0.939705
1,RandomForest,0.650323,0.021721,0.108801,0.105064,0.933499
5,SVM,0.599641,0.045447,0.097603,0.000000,0.950426
3,DecisionTree,0.568831,0.038167,0.072713,0.084674,0.925624
